# Test Trace Answer Visual-Index Analysis

Compare test accuracy when every mined `PATCH`/`REGION` action is replayed with either a random visual index or the final available visual index. Each trace is evaluated by the model that mined it.

In [ ]:
from pathlib import Path
import json
import re

import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
import pandas as pd
from IPython.display import display

ROOT = next(
    (candidate for candidate in (Path.cwd(), *Path.cwd().parents) if (candidate / "scripts/eval_mined_traces_m3cot.py").exists()),
    Path.cwd(),
)

RESULTS_ROOT = ROOT / "outputs/inference/test_oracle_visual_index"
MODELS = ("ivtlr", "lvar")
CONTEXTS = ("global", "coarse")
INDEX_MODES = ("random", "last")
RESULTS_ROOT

In [ ]:
def prediction_paths(model: str, context: str, index_mode: str) -> list[Path]:
    run_dir = (
        RESULTS_ROOT
        / f"mined_by_{model}_ckpt"
        / f"evaluated_by_{model}_ckpt"
        / f"context_{context}"
        / f"visual_index_mode_{index_mode}"
    )
    pattern = f"m3cot_test_predictions_mined-by_{model}_evaluated-by_{model}_{context}_raw_{index_mode}*.jsonl"
    return sorted(run_dir.glob(pattern))


def replay_context_from_path(path: Path, mining_context: str) -> str:
    match = re.search(r"_replayed-under_(global|coarse|full_context|global_mean)(?:_|\.)", path.name)
    return match.group(1) if match else mining_context


def read_jsonl(path: Path) -> list[dict]:
    with path.open(encoding="utf-8") as handle:
        return [json.loads(line) for line in handle if line.strip()]


records = []
for model in MODELS:
    for context in CONTEXTS:
        for index_mode in INDEX_MODES:
            paths = prediction_paths(model, context, index_mode)
            if not paths:
                records.append({
                    "model": model.upper(),
                    "context": context,
                    "replay_context": context,
                    "visual_index_mode": index_mode,
                    "correct": None,
                    "total": 0,
                    "accuracy": None,
                    "status": "missing",
                    "path": "not found",
                })
                continue

            for path in paths:
                rows = read_jsonl(path)
                correct = sum(bool(row.get("correct")) for row in rows)
                total = len(rows)
                records.append({
                    "model": model.upper(),
                    "context": context,
                    "replay_context": replay_context_from_path(path, context),
                    "visual_index_mode": index_mode,
                    "correct": correct,
                    "total": total,
                    "accuracy": correct / total if total else None,
                    "status": "complete" if total else "empty",
                    "path": str(path),
                })

results = pd.DataFrame(records)
display(results.style.format({"accuracy": "{:.2%}"}))

## Accuracy by setting

In [ ]:
accuracy_table = results.pivot_table(
    index=["model", "context", "replay_context"],
    columns="visual_index_mode",
    values="accuracy",
    aggfunc="first",
    dropna=False,
).reindex(columns=list(INDEX_MODES))
accuracy_table.columns.name = "visual index mode"
display(accuracy_table.style.format("{:.2%}").highlight_max(axis=1, color="#d8f3dc"))

In [ ]:
plot_data = accuracy_table.dropna(how="all")
if plot_data.empty:
    print(f"No completed visual-index results found under {RESULTS_ROOT}")
else:
    ax = plot_data.plot(kind="bar", figsize=(10, 5), rot=0, color=["#4C78A8", "#F58518"])
    ax.set_title("Mined-trace accuracy by visual-index setting")
    ax.set_xlabel("Model and context")
    ax.set_ylabel("Accuracy")
    ax.set_ylim(0, 1)
    ax.yaxis.set_major_formatter(PercentFormatter(1.0))
    ax.legend(title="Visual index mode")
    ax.grid(axis="y", alpha=0.25)
    plt.tight_layout()
    plt.show()

## Missing or empty runs

In [ ]:
incomplete = results[results["status"] != "complete"]
if incomplete.empty:
    print("All expected visual-index settings have at least one result.")
else:
    display(incomplete[["model", "context", "replay_context", "visual_index_mode", "status", "path"]])